## Chain Rules


### Pytorch Example


In [13]:
import torch

x = torch.tensor(2.0, requires_grad=True)

y = x ** 2 + 3 * x + 1

y.backward()

print(x.grad)

tensor(7.)


## Build your Own

### Value Definition
* Value Class
* Mathmatical Operations with gradient trace
* Back propagate

In [14]:
from torch.utils import data


class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._pre = set(_children)
        self._op = _op

    def __repr__(self) -> str:
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if out.data > 0 else 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._pre:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    ### Other calculations
    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, n):
        assert isinstance(n, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * self.data ** (n - 1) * out.grad

        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

    def exp(self):
        import math
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        import math
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += out.grad / self.data
        out._backward = _backward
        return out

    def tanh(self):
        import math
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out
    
    
    
    

    

### MLP from Scratch

In [15]:
import random

class Neuron:
    def __init__(self, n_inputs):
        self.w = [
            Value(random.uniform(-1, 1)) for _ in range(n_inputs)
        ]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_inputs, n_outputs):
        self.neurons = [
            Neuron(n_inputs) for _ in range(n_outputs)
        ]
    
    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):
        self.layers = [
            Layer(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


### XOR Training

In [16]:
random.seed(42)

model = MLP([2, 4, 1])

xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [-1, 1, 1, -1]   # XOR pattern

for step in range(100):
    predictions = [model(x) for x in xs]
    loss = sum((p - y)** 2 for p, y in zip(predictions, ys))

    for p in model.parameters():
        p.grad = 0.0
    loss.backward()

    learing_ratio = 0.05
    for p in model.parameters():
        p.data -= learing_ratio * p.grad

    if step % 20 == 0:
        print(f"step {step:3d}, loss = {loss.data:.4f}")

print("\n Predictions after training: ")
for x, y in zip(xs, ys):
    print(f"   input={x}, target={y:2d} pred={model(x).data:6.3f}")


step   0, loss = 4.1491
step  20, loss = 2.9166
step  40, loss = 1.4733
step  60, loss = 0.6011
step  80, loss = 0.2936

 Predictions after training: 
   input=[0, 0], target=-1 pred=-0.853
   input=[0, 1], target= 1 pred= 0.771
   input=[1, 0], target= 1 pred= 0.801
   input=[1, 1], target=-1 pred=-0.753


## Gradient Check

In [17]:
def gradient_check(build_expr, x_val, h=1e-7):
    x = Value(x_val)
    y = build_expr(x)
    y.backward()
    autodiff_grad = x.grad

    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical_grad = (y_plus - y_minus) / (2 * h)

    diff = abs(autodiff_grad - numerical_grad)
    return autodiff_grad, numerical_grad, diff


def expr(x):
    return (x ** 3 + 2 * 2  +1).tanh()

ad, num, diff = gradient_check(expr, 0.5)
print(f"Autodiff grad: {ad:.8f}, Numerical grad: {num:.8f}, Diff: {diff:.4f}")

Autodiff grad: 0.00010607, Numerical grad: 0.00010607, Diff: 0.0000


## Some Partice